# FHIR-Aggregator
## Explore data in the test google fhir service

### install and test dependencies

In [ ]:
pip install dtale

# install the query tool

In [ ]:
pip install git+https://github.com/FHIR-Aggregator/fhir-query.git

# verify the tool was installed

In [ ]:
!fq

# retrieve vocabularies used on commonly used resources

In [ ]:
%env FHIR_BASE=https://google-fhir.fhir-aggregator.org
!fq vocabulary vocabulary.tsv --fhir-base-url $FHIR_BASE

### show vocabularies

In [ ]:
import pandas as pd
import dtale.app as dtale_app
import dtale
df = pd.read_csv('vocabulary.tsv', sep='\t')
dtale_app.USE_COLAB = True
dtale.show(df)

# retrieve a pre-defined set of queries, a GraphDefinition
## in this case, retrieve an entire study

In [ ]:
!wget https://raw.githubusercontent.com/FHIR-Aggregator/fhir-query/refs/heads/main/graph-definitions/R5/ResearchStudyGraph.yaml


# export the data to a local database

In [ ]:
%env  FHIR_BASE=https://google-fhir.fhir-aggregator.org
# export a study using a set of stored queries
!fq --fhir-base-url $FHIR_BASE  --graph-definition-file-path  ResearchStudyGraph.yaml  --path '/ResearchStudy?identifier=TCGA-KIRC'

In [ ]:
# summarize the extracted data
!fq summarize

In [ ]:
# create a dataframe from  the extracted data
!fq dataframe


In [ ]:
import pandas as pd
import dtale.app as dtale_app
import dtale
df = pd.read_csv('/tmp/fhir-graph.tsv')
dtale_app.USE_COLAB = True
dtale.show(df)

In [ ]:
pip install lifelines

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

df['days_follow_up'] = (
    df['patient_observation_number_of_days_between_index_date_and_last_follow_up']
    .str.replace(' days', '', regex=False)
    .replace('', np.nan) 
    .astype(float)   
)
median_days = df['days_follow_up'].median()
df['days_follow_up'] = df['days_follow_up'].fillna(median_days).astype(int) # Shortened column name and converted values to int days

df_unique = df.drop_duplicates(subset=['patient_id']) # This is the primary key for a transformed FHIR patient/participant

T = df_unique['days_follow_up']
E = df_unique['patient_deceasedBoolean'].astype(bool) 

kmf = KaplanMeierFitter()
kmf.fit(T, event_observed=E)

plt.figure(figsize=(10, 6))
kmf.plot_survival_function()
plt.title('Kaplan-Meier Survival Curve')
plt.xlabel('Days Since Index Date')
plt.ylabel('Survival Probability')
plt.grid()
plt.show()
